# Industrial Pack - Eurotherm Setpoints

This notebook shows how to:
1. Connect to SPX with `spx-python`
2. Ensure Eurotherm instances are running
3. Update temperature and pressure setpoints

Prerequisites:
- SPX server running at `http://localhost:8000`
- Industrial Pack installed (profile `process_cell_quickstart`)
- `SPX_PRODUCT_KEY` set in your environment
- Instance keys: `spx_eurotherm_3216_temp`, `spx_eurotherm_3504_pressure`


In [1]:
import os
import spx_python

SPX_API_URL = os.environ.get("SPX_API_URL", "http://localhost:8000")
PRODUCT_KEY = os.environ.get("SPX_PRODUCT_KEY")
EURO_3216_INSTANCE = "spx_eurotherm_3216_temp"
EURO_3504_INSTANCE = "spx_eurotherm_3504_pressure"

if not PRODUCT_KEY:
    raise ValueError("SPX_PRODUCT_KEY is required.")

client = spx_python.init(address=SPX_API_URL, product_key=PRODUCT_KEY)
print(f"Connected to {SPX_API_URL}")


Connected to http://localhost:8000


In [2]:
def ensure_running(instance_key: str):
    try:
        instance = client["instances"][instance_key]
    except Exception as exc:
        raise RuntimeError(
            f"Instance {instance_key!r} not found. Start the Industrial Pack first."
        ) from exc

    state = instance.state
    print(f"{instance_key}: {state}")
    if (state or "").lower() != "running":
        instance.start()
        print(f"{instance_key}: started")
    return instance

euro_3216 = ensure_running(EURO_3216_INSTANCE)
euro_3504 = ensure_running(EURO_3504_INSTANCE)


spx_eurotherm_3216_temp: RUNNING
spx_eurotherm_3504_pressure: RUNNING


In [5]:
attrs_3216 = euro_3216["attributes"]
attrs_3504 = euro_3504["attributes"]

temp_sp = attrs_3216["target_sp_raw"]
pressure_sp = attrs_3504["pressure_sp_bar"]

# Ensure auto mode (0 = auto, 1 = manual)
attrs_3216["auto_man_raw"].internal_value = 0
attrs_3504["auto_man_raw"].internal_value = 0

print("Before update:")
print("  Eurotherm 3216 setpoint_c =", temp_sp.external_value)
print("  Eurotherm 3504 pressure_sp_bar =", pressure_sp.external_value)

temp_sp.internal_value = 60.0
pressure_sp.internal_value = 2.5

print("After update:")
print("  Eurotherm 3216 setpoint_c =", temp_sp.external_value)
print("  Eurotherm 3504 pressure_sp_bar =", pressure_sp.external_value)


Before update:
  Eurotherm 3216 setpoint_c = 30
  Eurotherm 3504 pressure_sp_bar = 10.0
After update:
  Eurotherm 3216 setpoint_c = 60
  Eurotherm 3504 pressure_sp_bar = 10.0
